In [1]:
import ast
import re
import warnings
import random
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

In [2]:
SEED = 322
N_FOLDS = 5
N_LABELS = 5
TRAIN_PATH = "/Users/sofkatulyakova/Desktop/Го дз3-кагл/train.csv"
TEST_PATH = "/Users/sofkatulyakova/Desktop/Го дз3-кагл/test.csv"
OUTPUT_PATH = "/Users/sofkatulyakova/Desktop/Го дз3-кагл/sample_submission.csv"

random.seed(SEED)
np.random.seed(SEED)

In [3]:
_RE_HTML = re.compile(r"<[^>]+>", re.S)
_RE_CDATA = re.compile(r"<!\[CDATA\[.*?\]\]>", re.S)
_RE_ENTITY = re.compile(r"&[a-zA-Z#0-9]+;")
_RE_SPACES = re.compile(r"\s+")
_RE_EMOJI = re.compile(
    "[\U0001F300-\U0001F9FF\U00002600-\U000027BF\U0001FA00-\U0001FAFF]+",
    flags=re.UNICODE,
)
def clean_text(t: str) -> str:
    if not isinstance(t, str):
        return ""
    t = _RE_CDATA.sub(" ", t)
    t = _RE_HTML.sub(" ", t)
    t = _RE_ENTITY.sub(" ", t)
    t = _RE_EMOJI.sub(" ", t)
    return _RE_SPACES.sub(" ", t).strip()

In [4]:
def multilabel_stratified_kfold(y, n_splits=5, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y)
    order = rng.permutation(n)
    fold_counts = np.zeros((n_splits, y.shape[1]), dtype=np.float64)
    assignments = np.empty(n, dtype=np.int32)
    for i in order:
        yi = y[i]
        desired = (fold_counts.sum(axis=0) + 1e-9) / n_splits
        scores = np.abs(fold_counts - desired * (fold_counts + 1)).sum(axis=1)
        scores += rng.rand(n_splits) * 1e-6
        assignments[i] = np.argmin(scores)
        fold_counts[assignments[i]] += yi
    for k in range(n_splits):
        yield np.where(assignments != k)[0], np.where(assignments == k)[0]

def hamming_loss(y_true, y_pred):
    return np.mean(y_true != y_pred)

In [ ]:
def main():
    train = pd.read_csv(TRAIN_PATH, sep="\t")
    test  = pd.read_csv(TEST_PATH,  sep="\t")
    train["target"] = train["target"].apply(ast.literal_eval)
    y = np.array(train["target"].tolist(), dtype=np.float32)

    train_text = (train["title"].fillna("").apply(clean_text) + " [SEP] " +
                  train["text"].fillna("").apply(clean_text))
    test_text  = (test["title"].fillna("").apply(clean_text)  + " [SEP] " +
                  test["text"].fillna("").apply(clean_text))
    train_title = train["title"].fillna("").apply(clean_text)
    test_title  = test["title"].fillna("").apply(clean_text)

    tfidf = TfidfVectorizer(max_features=80000, ngram_range=(1, 2), min_df=2,
                            max_df=0.97, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
    tfidf.fit(pd.concat([train_text, test_text], ignore_index=True))
    X_tr_full = tfidf.transform(train_text)
    X_te_full = tfidf.transform(test_text)

    tfidf_t = TfidfVectorizer(max_features=20000, ngram_range=(1, 3), min_df=2,
                              max_df=0.97, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
    tfidf_t.fit(pd.concat([train_title, test_title], ignore_index=True))
    X_tr_title = tfidf_t.transform(train_title)
    X_te_title = tfidf_t.transform(test_title)

    tfidf_char = TfidfVectorizer(analyzer='char_wb', max_features=50000,
                                 ngram_range=(3, 5), min_df=3, sublinear_tf=True)
    tfidf_char.fit(pd.concat([train_text, test_text], ignore_index=True))
    X_tr_char = tfidf_char.transform(train_text)
    X_te_char = tfidf_char.transform(test_text)

    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    src_tr = ohe.fit_transform(train[["source"]].fillna("unknown"))
    src_te = ohe.transform(test[["source"]].fillna("unknown"))

    def build_date_features_scaled(df_tr, df_te):
        def extract(df):
            dt = pd.to_datetime(df["publication_date"], errors="coerce")
            return np.column_stack([
                dt.dt.year.fillna(2019).astype(np.float32),
                dt.dt.month.fillna(1).astype(np.float32),
                dt.dt.dayofweek.fillna(0).astype(np.float32),
                dt.dt.hour.fillna(0).astype(np.float32),
            ])
        scaler = StandardScaler()
        return csr_matrix(scaler.fit_transform(extract(df_tr))), \
               csr_matrix(scaler.transform(extract(df_te)))

    date_tr, date_te = build_date_features_scaled(train, test)

    embedder = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
    
    train_emb = embedder.encode(
        train_text.tolist(), batch_size=64,
        show_progress_bar=True, normalize_embeddings=True
    )
    test_emb = embedder.encode(
        test_text.tolist(), batch_size=64,
        show_progress_bar=True, normalize_embeddings=True
    )
    emb_tr = csr_matrix(train_emb)
    emb_te = csr_matrix(test_emb)

    train_title_emb = embedder.encode(
        train_title.tolist(), batch_size=64,
        show_progress_bar=True, normalize_embeddings=True
    )
    test_title_emb = embedder.encode(
        test_title.tolist(), batch_size=64,
        show_progress_bar=True, normalize_embeddings=True
    )
    emb_title_tr = csr_matrix(train_title_emb)
    emb_title_te = csr_matrix(test_title_emb)

    X_train = hstack([X_tr_full, X_tr_title, X_tr_char, src_tr, date_tr, emb_tr, emb_title_tr])
    X_test  = hstack([X_te_full, X_te_title, X_te_char, src_te, date_te, emb_te, emb_title_te])

    oof_probs  = np.zeros((len(train), N_LABELS), dtype=np.float32)
    test_probs = np.zeros((len(test),  N_LABELS), dtype=np.float32)
    splits = list(multilabel_stratified_kfold(y, N_FOLDS, SEED))

    base_models = [
        LogisticRegression(C=1.0, max_iter=300, solver="lbfgs", random_state=SEED),
        LogisticRegression(C=4.0, max_iter=300, solver="lbfgs", random_state=SEED),
        RidgeClassifier(alpha=1.0, random_state=SEED),
    ]

    for fold, (tr_idx, va_idx) in enumerate(splits):
        X_tr_f, y_tr_f = X_train[tr_idx], y[tr_idx]
        X_va_f         = X_train[va_idx]

        fold_test_probs = np.zeros((len(base_models), len(test), N_LABELS))

        for m_idx, base_clf in enumerate(base_models):
            calibrated_ovr = OneVsRestClassifier(
                CalibratedClassifierCV(estimator=base_clf, cv=3, method='sigmoid')
            )
            calibrated_ovr.fit(X_tr_f, y_tr_f)
            oof_probs[va_idx]      += calibrated_ovr.predict_proba(X_va_f) / len(base_models)
            fold_test_probs[m_idx]  = calibrated_ovr.predict_proba(X_test)

        test_probs += fold_test_probs.mean(axis=0)

    test_probs /= N_FOLDS

    thresholds = np.full(N_LABELS, 0.5)
    for lbl in range(N_LABELS):
        best_hl, best_thr = 1.0, 0.5
        for thr in np.arange(0.05, 0.96, 0.01):
            preds = (oof_probs[:, lbl] >= thr).astype(int)
            hl = hamming_loss(y[:, lbl], preds)
            if hl < best_hl:
                best_hl, best_thr = hl, thr
        thresholds[lbl] = best_thr


    test_bin = (test_probs >= thresholds).astype(int)
    sub = pd.DataFrame({
        "id":     test["id"].values,
        "target": [row.tolist() for row in test_bin]
    })
    sub.to_csv(OUTPUT_PATH, index=False)

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/261 [00:00<?, ?it/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]

Batches:   0%|          | 0/261 [00:00<?, ?it/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]